In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings

# Matikan warning biar output bersih
warnings.filterwarnings('ignore')

# ==========================================
# PART 1: LOAD, CLEAN & PREPARE DATA
# ==========================================
filename = 'm67.tsv' # Pastikan nama file sesuai
print(f"🚀 Loading data from {filename}...")

try:
    # Load data (VizieR biasanya pakai semicolon ';')
    df = pd.read_csv(filename, sep=';', comment='#', on_bad_lines='skip')
    
    # 1. STANDARDIZE COLUMN NAMES
    # Kita ubah nama kolom VizieR jadi standar codingan (ra, dec, pmra, etc.)
    # Ini PENTING biar script UPMASK di bawah gak bingung
    rename_map = {
        'RA_ICRS': 'ra', 'DE_ICRS': 'dec',
        'Plx': 'parallax', 'e_Plx': 'parallax_error',
        'pmRA': 'pmra', 'e_pmRA': 'pmra_error',
        'pmDE': 'pmdec', 'e_pmDE': 'pmdec_error',
        'Gmag': 'phot_g_mean_mag', 'BP-RP': 'bp_rp',
        'RUWE': 'ruwe'
    }
    df.rename(columns=rename_map, inplace=True)
    
    # 2. FORCE NUMERIC (Data Cleaning)
    # Kolom yang wajib angka (kadang VizieR nyelip string 'mas' atau null)
    numeric_cols = ['ra', 'dec', 'pmra', 'pmdec', 'parallax', 'ruwe', 
                    'parallax_error', 'pmra_error', 'pmdec_error']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 3. DROP NAN
    # Buang baris yang astrometrinya bolong
    df_clean = df.dropna(subset=['ra', 'dec', 'pmra', 'pmdec', 'parallax', 'parallax_error'])
    
    print(f"✅ Data Cleaned: {len(df_clean)} stars (Original: {len(df)})")
    
except Exception as e:
    print(f"❌ Error loading/cleaning data: {e}")
    raise # Stop execution if data fails

# ==========================================
# PART 2: INITIAL VISUALIZATION (SANITY CHECK)
# ==========================================
plt.figure(figsize=(12, 5))

# Plot 1: Sky Map
plt.subplot(1, 2, 1)
plt.scatter(df_clean['ra'], df_clean['dec'], s=1, c='black', alpha=0.1)
plt.title(f"Initial Spatial Map (N={len(df_clean)})")
plt.xlabel("RA (deg)"); plt.ylabel("Dec (deg)")
plt.gca().invert_xaxis()

# Plot 2: VPD (Vector Point Diagram)
plt.subplot(1, 2, 2)
plt.scatter(df_clean['pmra'], df_clean['pmdec'], s=1, c='black', alpha=0.1)
plt.xlim(-25, 5); plt.ylim(-15, 5) # Zoom M67 area
plt.title("Vector Point Diagram (Pre-Selection)")
plt.xlabel("PMRA (mas/yr)"); plt.ylabel("PMDEC (mas/yr)")
plt.grid(True, alpha=0.3)
plt.axvline(-10.9, c='red', ls='--', lw=0.5) # Target center reference
plt.axhline(-2.9, c='red', ls='--', lw=0.5)

plt.tight_layout()
plt.show()

# ==========================================
# PART 3: UPMASK LITE ALGORITHM
# ==========================================
# Configuration for M67
ITERATIONS = 50       # Monte Carlo cycles
K_CLUSTERS = 10       # Number of K-Means clusters
TARGET_PMRA = -10.9   # M67 Literature PMRA
TARGET_PMDEC = -2.9   # M67 Literature PMDEC
TARGET_PARALLAX = 1.13 # M67 Literature Parallax

print(f"\n🚀 Starting UPMASK Algorithm ({ITERATIONS} iterations)...")

X_cols = ['ra', 'dec', 'parallax', 'pmra', 'pmdec']
membership_votes = np.zeros(len(df_clean))

for i in range(ITERATIONS):
    if i % 10 == 0: print(f"  ... Iteration {i}/{ITERATIONS}")
    
    # A. Resampling (Add Gaussian Noise based on Errors)
    X_noisy = df_clean[X_cols].copy()
    X_noisy['parallax'] += df_clean['parallax_error'] * np.random.randn(len(df_clean))
    X_noisy['pmra']     += df_clean['pmra_error'] * np.random.randn(len(df_clean))
    X_noisy['pmdec']    += df_clean['pmdec_error'] * np.random.randn(len(df_clean))
    
    # B. Scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_noisy)
    
    # C. Clustering (K-Means)
    kmeans = KMeans(n_clusters=K_CLUSTERS, n_init=1, random_state=i)
    labels = kmeans.fit_predict(X_scaled)
    
    # D. Identify "The Cluster" (Closest to Target)
    centers = kmeans.cluster_centers_
    centers_original = scaler.inverse_transform(centers)
    
    # Distance to target (weighted parallax)
    dist_sq = (centers_original[:, 3] - TARGET_PMRA)**2 + \
              (centers_original[:, 4] - TARGET_PMDEC)**2 + \
              (centers_original[:, 2] - TARGET_PARALLAX)**2 * 10
              
    best_cluster_id = np.argmin(dist_sq)
    
    # E. Vote
    membership_votes[labels == best_cluster_id] += 1

# Calculate Probability
df_clean['prob'] = membership_votes / ITERATIONS
print("✅ UPMASK Completed!")

# ==========================================
# PART 4: FINAL RESULTS & PLOTTING
# ==========================================
# Filter Members (Threshold P >= 0.5)
members = df_clean[df_clean['prob'] >= 0.5]
field = df_clean[df_clean['prob'] < 0.5]

print(f"\n📊 Final Statistics:")
print(f"   Total Stars Processed: {len(df_clean)}")
print(f"   Likely Members (P>=0.5): {len(members)} stars")
print(f"   Field Contaminants: {len(field)} stars")

# Save to CSV
output_filename = "M67_Members_Final.csv"
members.to_csv(output_filename, index=False)
print(f"💾 Results saved to: {output_filename}")

# Final Plots (English Labels)
plt.figure(figsize=(12, 6))

# Plot 1: Sky Map (Members vs Field)
plt.subplot(1, 2, 1)
plt.scatter(field['ra'], field['dec'], s=1, c='gray', alpha=0.1, label='Field Stars')
plt.scatter(members['ra'], members['dec'], s=5, c='red', alpha=0.8, label='Cluster Members')
plt.title(f"Membership Spatial Distribution (N={len(members)})")
plt.xlabel("RA (deg)"); plt.ylabel("Dec (deg)")
plt.legend(loc='upper right')
plt.gca().invert_xaxis()

# Plot 2: VPD (Verification)
plt.subplot(1, 2, 2)
plt.scatter(field['pmra'], field['pmdec'], s=1, c='gray', alpha=0.1)
plt.scatter(members['pmra'], members['pmdec'], s=5, c='red', alpha=0.6)
plt.xlim(-25, 5); plt.ylim(-15, 5)
plt.title("Membership in Proper Motion Space")
plt.xlabel("PMRA (mas/yr)"); plt.ylabel("PMDEC (mas/yr)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quick Histogram of Probabilities
plt.figure(figsize=(6, 4))
plt.hist(df_clean['prob'], bins=20, color='skyblue', edgecolor='black')
plt.title("Membership Probability Distribution")
plt.xlabel("Probability (P)")
plt.ylabel("Count")
plt.show()

ini sesuaikan untuk konfigurasi + tsvnya